# --------------------------- Data Preparation for Regression ---------------------------

### Importing important packages

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import math as math
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

from sklearn.model_selection import train_test_split

## Importing cleaned datasets

In [2]:
# ============================================================
# LOAD CLEANED DATA -- previously saved in pickle file
# ============================================================

cleaned_data = pd.read_pickle(
    "EuroEats_cleaned_data.pkl"
)

customers = cleaned_data["customers"]
restaurants = cleaned_data["restaurants"]
menu_items = cleaned_data["menu_items"]
drivers = cleaned_data["drivers"]
orders = cleaned_data["orders"]
order_items = cleaned_data["order_items"]
deliveries = cleaned_data["deliveries"]
payments = cleaned_data["payments"]
reviews = cleaned_data["reviews"]

print("Cleaned EuroEats data loaded successfully.")

Cleaned EuroEats data loaded successfully.


In [3]:
# ============================================================
# BLOCK 2 — DEFINE ML PROBLEM
#
# Business Problem:
# Predict the actual delivery time of an order.
#
# Target:
# actual_delivery_minutes
#
# Problem Type:
# Regression
# ============================================================

target = "actual_delivery_minutes"

print("Target variable:", target)
print("Problem type: Regression")

Target variable: actual_delivery_minutes
Problem type: Regression


In [4]:
# ============================================================
# BLOCK 3 — CREATE BASE ML DATASET
# ============================================================

ml_data = (
    deliveries[
        [
            "order_id",
            "driver_id",
            "distance_km",
            "traffic_level",
            "weather_condition",
            "temperature_c",
            "pickup_delay_minutes",
            "restaurant_wait_minutes",
            "estimated_delivery_minutes",
            "actual_delivery_minutes"
        ]
    ]

    # Order information
    .merge(
        orders[
            [
                "order_id",
                "customer_id",
                "restaurant_id",
                "order_datetime",
                "number_of_items",
                "subtotal_eur",
                "delivery_fee_eur",
                "discount_eur",
                "payment_method"
            ]
        ],
        on="order_id",
        how="left"
    )

    # Restaurant information
    .merge(
        restaurants[
            [
                "restaurant_id",
                "country",
                "city",
                "cuisine_type",
                "price_level",
                "restaurant_rating",
                "average_preparation_time",
                "restaurant_size",
                "average_daily_orders"
            ]
        ],
        on="restaurant_id",
        how="left"
    )

    # Customer information
    .merge(
        customers[
            [
                "customer_id",
                "age",
                "gender",
                "customer_segment",
                "loyalty_status"
            ]
        ],
        on="customer_id",
        how="left"
    )

    # Driver information
    .merge(
        drivers[
            [
                "driver_id",
                "vehicle_type",
                "years_experience",
                "driver_rating",
                "completed_deliveries",
                "employment_type",
                "average_speed_kmh"
            ]
        ],
        on="driver_id",
        how="left"
    )
)

print("ML dataset shape:", ml_data.shape)

ml_data.head()

ML dataset shape: (1000, 36)


,order_id,driver_id,distance_km,traffic_level,weather_condition,temperature_c,pickup_delay_minutes,restaurant_wait_minutes,estimated_delivery_minutes,actual_delivery_minutes,...,age,gender,customer_segment,loyalty_status,vehicle_type,years_experience,driver_rating,completed_deliveries,employment_type,average_speed_kmh
0,1687,101,2.48,Medium,Rain,11.5,5.4,6.8,19.3,16.8,...,32,Female,Budget,Platinum,Car,1.1,4.15,629,Full-Time,30.4
1,4194,68,1.23,High,Cloudy,14.8,8.5,26.2,41.2,46.3,...,27,Female,Budget,Silver,Motorcycle,0.4,3.29,272,Part-Time,35.6
2,1541,273,1.58,Medium,Clear,27.3,10.2,16.5,32.5,33.6,...,36,Male,Regular,Bronze,E-Bike,0.3,3.69,168,Part-Time,25.2
3,2204,285,2.05,Medium,Clear,19.6,3.2,13.6,26.1,28.9,...,18,Female,Regular,Platinum,Bicycle,1.9,3.75,830,Part-Time,14.6
4,3613,293,2.10,Low,Rain,1.2,2.5,10.1,16.5,16.0,...,40,Male,Budget,Gold,Scooter,0.1,3.52,172,Freelance,38.1


In [5]:
# ============================================================
# BLOCK 3 — CREATE BASE ML DATASET
# ============================================================

ml_data = (
    deliveries[
        [
            "order_id",
            "driver_id",
            "distance_km",
            "traffic_level",
            "weather_condition",
            "temperature_c",
            "pickup_delay_minutes",
            "restaurant_wait_minutes",
            "estimated_delivery_minutes",
            "actual_delivery_minutes"
        ]
    ]

    # Order information
    .merge(
        orders[
            [
                "order_id",
                "customer_id",
                "restaurant_id",
                "order_datetime",
                "number_of_items",
                "subtotal_eur",
                "delivery_fee_eur",
                "discount_eur",
                "payment_method"
            ]
        ],
        on="order_id",
        how="left"
    )

    # Restaurant information
    .merge(
        restaurants[
            [
                "restaurant_id",
                "country",
                "city",
                "cuisine_type",
                "price_level",
                "restaurant_rating",
                "average_preparation_time",
                "restaurant_size",
                "average_daily_orders"
            ]
        ],
        on="restaurant_id",
        how="left"
    )

    # Customer information
    .merge(
        customers[
            [
                "customer_id",
                "age",
                "gender",
                "customer_segment",
                "loyalty_status"
            ]
        ],
        on="customer_id",
        how="left"
    )

    # Driver information
    .merge(
        drivers[
            [
                "driver_id",
                "vehicle_type",
                "years_experience",
                "driver_rating",
                "completed_deliveries",
                "employment_type",
                "average_speed_kmh"
            ]
        ],
        on="driver_id",
        how="left"
    )
)

print("ML dataset shape:", ml_data.shape)

ml_data.head()

ML dataset shape: (1000, 36)


,order_id,driver_id,distance_km,traffic_level,weather_condition,temperature_c,pickup_delay_minutes,restaurant_wait_minutes,estimated_delivery_minutes,actual_delivery_minutes,...,age,gender,customer_segment,loyalty_status,vehicle_type,years_experience,driver_rating,completed_deliveries,employment_type,average_speed_kmh
0,1687,101,2.48,Medium,Rain,11.5,5.4,6.8,19.3,16.8,...,32,Female,Budget,Platinum,Car,1.1,4.15,629,Full-Time,30.4
1,4194,68,1.23,High,Cloudy,14.8,8.5,26.2,41.2,46.3,...,27,Female,Budget,Silver,Motorcycle,0.4,3.29,272,Part-Time,35.6
2,1541,273,1.58,Medium,Clear,27.3,10.2,16.5,32.5,33.6,...,36,Male,Regular,Bronze,E-Bike,0.3,3.69,168,Part-Time,25.2
3,2204,285,2.05,Medium,Clear,19.6,3.2,13.6,26.1,28.9,...,18,Female,Regular,Platinum,Bicycle,1.9,3.75,830,Part-Time,14.6
4,3613,293,2.10,Low,Rain,1.2,2.5,10.1,16.5,16.0,...,40,Male,Budget,Gold,Scooter,0.1,3.52,172,Freelance,38.1


In [6]:
# ============================================================
# BLOCK 4 — CHECK TARGET VARIABLE
# ============================================================

ml_data[target] = pd.to_numeric(
    ml_data[target],
    errors="coerce"
)

print("Target missing values:", ml_data[target].isna().sum())

print("\nTarget statistics:")
print(
    ml_data[target].describe()
)

Target missing values: 0

Target statistics:
count    1000.00000
mean       41.35150
std        19.78546
min         9.30000
25%        28.60000
50%        36.65000
75%        47.90000
max       130.00000
Name: actual_delivery_minutes, dtype: float64


In [7]:
# ============================================================
# BLOCK 6 — SELECT ML FEATURES
# ============================================================

feature_columns = [
    # Order characteristics
    "number_of_items",
    "subtotal_eur",
    "delivery_fee_eur",
    "discount_eur",

    # Delivery conditions
    "distance_km",
    "traffic_level",
    "weather_condition",
    "temperature_c",
    "estimated_delivery_minutes",

    # Restaurant characteristics
    "country",
    "city",
    "cuisine_type",
    "price_level",
    "restaurant_rating",
    "average_preparation_time",
    "restaurant_size",
    "average_daily_orders",

    # Customer characteristics
    "age",
    "gender",
    "customer_segment",
    "loyalty_status",

    # Driver characteristics
    "vehicle_type",
    "years_experience",
    "driver_rating",
    "completed_deliveries",
    "employment_type",
    "average_speed_kmh"
]

X = ml_data[feature_columns].copy()

y = ml_data[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 27)
y shape: (1000,)


In [8]:
# ============================================================
# BLOCK 8 — REMOVE MISSING TARGET VALUES
# ============================================================

valid_target = y.notna()

X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

print("X shape after target validation:", X.shape)
print("y shape after target validation:", y.shape)

print("\nMissing target values:", y.isna().sum())

X shape after target validation: (1000, 27)
y shape after target validation: (1000,)

Missing target values: 0


In [9]:
# ============================================================
# BLOCK 7 — NUMERIC DATA TYPE CONVERSION
# ============================================================

numeric_columns = [
    "number_of_items",
    "subtotal_eur",
    "delivery_fee_eur",
    "discount_eur",
    "distance_km",
    "temperature_c",
    "estimated_delivery_minutes",
    "restaurant_rating",
    "average_preparation_time",
    "average_daily_orders",
    "age",
    "years_experience",
    "driver_rating",
    "completed_deliveries",
    "average_speed_kmh"
]

for col in numeric_columns:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

y = pd.to_numeric(
    y,
    errors="coerce"
)

print("Numeric conversion completed.")

Numeric conversion completed.


In [10]:
# ============================================================
# BLOCK 9 — TARGET VALIDATION
# ============================================================

print("Minimum delivery time:", y.min())
print("Maximum delivery time:", y.max())
print("Mean delivery time:", round(y.mean(), 2))
print("Median delivery time:", round(y.median(), 2))

# Check impossible values
print(
    "\nZero/negative delivery times:",
    (y <= 0).sum()
)

Minimum delivery time: 9.3
Maximum delivery time: 130.0
Mean delivery time: 41.35
Median delivery time: 36.65

Zero/negative delivery times: 0


In [11]:
# ============================================================
# BLOCK 9 — TARGET VALIDATION
# ============================================================

print("Minimum delivery time:", y.min())
print("Maximum delivery time:", y.max())
print("Mean delivery time:", round(y.mean(), 2))
print("Median delivery time:", round(y.median(), 2))

# Check impossible values
print(
    "\nZero/negative delivery times:",
    (y <= 0).sum()
)

Minimum delivery time: 9.3
Maximum delivery time: 130.0
Mean delivery time: 41.35
Median delivery time: 36.65

Zero/negative delivery times: 0


In [12]:
# ============================================================
# BLOCK 10 — TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target :", y_test.shape)

Training data: (800, 27)
Testing data : (200, 27)

Training target: (800,)
Testing target : (200,)


In [13]:
# ============================================================
# BLOCK 11 — IDENTIFY NUMERIC & CATEGORICAL FEATURES
# ============================================================

numeric_features = [
    col for col in X_train.columns
    if X_train[col].dtype != "object"
]

categorical_features = [
    col for col in X_train.columns
    if X_train[col].dtype == "object"
]

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numeric features:",
      len(numeric_features))

print("Number of categorical features:",
      len(categorical_features))

Numeric features:
['number_of_items', 'subtotal_eur', 'delivery_fee_eur', 'discount_eur', 'distance_km', 'temperature_c', 'estimated_delivery_minutes', 'price_level', 'restaurant_rating', 'average_preparation_time', 'average_daily_orders', 'age', 'years_experience', 'driver_rating', 'completed_deliveries', 'average_speed_kmh']

Categorical features:
['traffic_level', 'weather_condition', 'country', 'city', 'cuisine_type', 'restaurant_size', 'gender', 'customer_segment', 'loyalty_status', 'vehicle_type', 'employment_type']

Number of numeric features: 16
Number of categorical features: 11


In [14]:
# ============================================================
# BLOCK 11 — IDENTIFY NUMERIC & CATEGORICAL FEATURES
# ============================================================

numeric_features = [
    col for col in X_train.columns
    if X_train[col].dtype != "object"
]

categorical_features = [
    col for col in X_train.columns
    if X_train[col].dtype == "object"
]

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numeric features:",
      len(numeric_features))

print("Number of categorical features:",
      len(categorical_features))

Numeric features:
['number_of_items', 'subtotal_eur', 'delivery_fee_eur', 'discount_eur', 'distance_km', 'temperature_c', 'estimated_delivery_minutes', 'price_level', 'restaurant_rating', 'average_preparation_time', 'average_daily_orders', 'age', 'years_experience', 'driver_rating', 'completed_deliveries', 'average_speed_kmh']

Categorical features:
['traffic_level', 'weather_condition', 'country', 'city', 'cuisine_type', 'restaurant_size', 'gender', 'customer_segment', 'loyalty_status', 'vehicle_type', 'employment_type']

Number of numeric features: 16
Number of categorical features: 11


In [15]:
# ============================================================
# BLOCK 13 — CATEGORICAL VALUE CHECK
# ============================================================

for col in categorical_features:

    print("\n" + "=" * 60)
    print(col)

    print(
        X_train[col]
        .value_counts(dropna=False)
        .head(20)
    )


traffic_level
traffic_level
High      301
Medium    271
Low       228
Name: count, dtype: int64

weather_condition
weather_condition
Clear     342
Cloudy    225
Rain      211
Snow       22
Name: count, dtype: int64

country
country
Germany        147
Italy          100
France          79
Spain           71
Poland          60
Belgium         52
Netherlands     51
Sweden          48
Portugal        37
Greece          36
Norway          33
Finland         26
Czechia         24
Austria         22
Denmark         14
Name: count, dtype: int64

city
city
Berlin       65
Rome         49
Stockholm    48
Paris        42
Brussels     37
Amsterdam    36
Warsaw       31
Lisbon       31
Munich       29
Barcelona    25
Hamburg      24
Madrid       24
Milan        24
Athens       22
Vienna       22
Helsinki     20
Oslo         19
Frankfurt    18
Krakow       16
Prague       15
Name: count, dtype: int64

cuisine_type
cuisine_type
Chinese          114
Spanish           87
Pizza             76
Local Cui

In [16]:
# ============================================================
# BLOCK 14 — NUMERIC FEATURE SUMMARY
# ============================================================

numeric_summary = (
    X_train[numeric_features]
    .describe()
    .T
)

numeric_summary["missing"] = (
    X_train[numeric_features]
    .isna()
    .sum()
)

numeric_summary

,count,mean,std,min,25%,50%,75%,max,missing
number_of_items,800.0,1.522500,0.777978,1.00,1.0000,1.000,2.0000,7.00,0
subtotal_eur,800.0,10.864287,8.842839,1.47,4.8575,8.335,13.9425,74.96,0
delivery_fee_eur,800.0,2.622450,0.484757,0.99,2.2900,2.630,2.9400,4.14,0
discount_eur,800.0,0.234813,0.659328,0.00,0.0000,0.000,0.0000,6.99,0
distance_km,800.0,3.668800,3.722109,0.40,1.3800,2.510,4.5425,28.00,0
temperature_c,800.0,12.912000,8.335199,-9.50,7.3000,12.600,18.8000,32.30,0
estimated_delivery_minutes,800.0,38.589625,19.780859,11.00,27.2000,34.450,44.1000,276.80,0
price_level,800.0,2.737500,1.102321,1.00,2.0000,3.000,4.0000,4.00,0
restaurant_rating,800.0,4.012750,0.429352,3.10,3.7000,4.000,4.3000,5.00,0
average_preparation_time,800.0,23.418750,5.901647,10.00,20.0000,24.000,27.0000,38.00,0


In [17]:
# ============================================================
# BLOCK 15 — FEATURE / TARGET CORRELATION
# ============================================================

correlation_data = (
    X_train[numeric_features]
    .copy()
)

correlation_data["actual_delivery_minutes"] = y_train

correlation = (
    correlation_data
    .corr(numeric_only=True)["actual_delivery_minutes"]
    .drop("actual_delivery_minutes")
    .sort_values(
        ascending=False
    )
)

print(correlation)

estimated_delivery_minutes    0.909683
distance_km                   0.727137
average_preparation_time      0.176374
average_daily_orders          0.149492
subtotal_eur                  0.136037
price_level                   0.115943
number_of_items               0.055250
age                           0.041224
discount_eur                  0.024132
delivery_fee_eur              0.009338
restaurant_rating             0.001689
completed_deliveries          0.000975
years_experience             -0.006154
driver_rating                -0.025761
temperature_c                -0.032683
average_speed_kmh            -0.263300
Name: actual_delivery_minutes, dtype: float64


In [18]:
# ============================================================
# BLOCK 16 — FINAL ML DATA PREPARATION CHECK
# ============================================================

print("==========================================")
print("EUROEATS ML DATA PREPARATION COMPLETE")
print("==========================================")

print("\nTraining features :", X_train.shape)
print("Testing features  :", X_test.shape)

print("Training target   :", y_train.shape)
print("Testing target    :", y_test.shape)

print("\nTarget:")
print("actual_delivery_minutes")

print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nTraining missing values:",
      X_train.isna().sum().sum())

print("Testing missing values:",
      X_test.isna().sum().sum())

EUROEATS ML DATA PREPARATION COMPLETE

Training features : (800, 27)
Testing features  : (200, 27)
Training target   : (800,)
Testing target    : (200,)

Target:
actual_delivery_minutes

Numeric features: 16
Categorical features: 11

Training missing values: 0
Testing missing values: 0


In [19]:
# ============================================================
# BLOCK 17 — REVISE FEATURES AFTER INITIAL MODEL EVALUATION
# ============================================================
#
# NOTE:
# After completing the initial model-building and evaluation,
# the feature "estimated_delivery_minutes" was reviewed again
# against the original EuroEats business problem.
#
# The reason for removing this feature is:
#
# 1. estimated_delivery_minutes is already the platform's
#    existing ETA shown to the customer.
#
# 2. The business objective is to build a regression model that
#    predicts actual delivery duration before dispatch.
#
# 3. Including the existing ETA would make the new model heavily
#    dependent on an existing prediction rather than independently
#    learning the operational factors that drive delivery time.
#
# 4. The initial feature-importance analysis showed that
#    estimated_delivery_minutes overwhelmingly dominated the model,
#    while the remaining operational variables had much smaller
#    importance.
#
# 5. Removing this feature provides a cleaner and more interpretable
#    model for understanding the underlying drivers of delivery time.
#
# Therefore, estimated_delivery_minutes will remain available for
# separate ETA accuracy/business analysis, but will NOT be used
# as an input feature in the final regression model.
#
# This block is intentionally placed after the initial ML preparation
# blocks because this feature decision was made after reviewing the
# first model results.
# ============================================================


# Remove the existing platform ETA from the training and test features
X_train = X_train.drop(
    columns=["estimated_delivery_minutes"],
    errors="ignore"
)

X_test = X_test.drop(
    columns=["estimated_delivery_minutes"],
    errors="ignore"
)


# Recreate numeric and categorical feature lists
# so all subsequent preprocessing/modeling blocks
# automatically use the revised feature set.

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()


# Display confirmation
print("Feature revision completed successfully.")
print("-" * 60)

print(
    "estimated_delivery_minutes present in X_train:",
    "estimated_delivery_minutes" in X_train.columns
)

print(
    "estimated_delivery_minutes present in X_test:",
    "estimated_delivery_minutes" in X_test.columns
)

print(f"\nNumber of features after revision: {X_train.shape[1]}")

print(f"Numeric features     : {len(numeric_features)}")
print(f"Categorical features : {len(categorical_features)}")

Feature revision completed successfully.
------------------------------------------------------------
estimated_delivery_minutes present in X_train: False
estimated_delivery_minutes present in X_test: False

Number of features after revision: 26
Numeric features     : 15
Categorical features : 11


In [20]:
# ============================================================
# BLOCK 18 — SAVE REVISED ML PREPARED DATA
# ============================================================
#
# The ML dataset has now been revised after the initial model
# evaluation. The existing platform ETA is excluded from the
# predictive feature set.
#
# Save the revised dataset using the SAME filename so that
# downstream notebooks do not require any code changes.
# ============================================================


ml_prepared_data = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}


pd.to_pickle(
    ml_prepared_data,
    "EuroEats_ML_prepared_data.pkl"
)


print("Revised ML prepared data saved successfully.")
print("File: EuroEats_ML_prepared_data.pkl")
print("estimated_delivery_minutes has been excluded from X_train and X_test.")

Revised ML prepared data saved successfully.
File: EuroEats_ML_prepared_data.pkl
estimated_delivery_minutes has been excluded from X_train and X_test.


###### ------------------------------------------------------------------------------------------------------------ THE END ------------------------------------------------------------------------------------------------------------